# Inference — self-contained, не потребує папки model/

In [ ]:
CKPT_PATHS = [
    'checkpoints/model_v1/fold0_epoch010_val0.6800.ckpt',
    # 'checkpoints/model_v1/fold1_...ckpt',
]
CSV_PATH    = 'data/test.csv'    # або train.csv
EEG_DIR     = 'data/test_eegs'  # або train_eegs
BACKBONE    = 'convnext_atto'   # має збігатись із тренуванням
BATCH_SIZE  = 32
NUM_WORKERS = 4
OUT_CSV     = 'submission.csv'


In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import butter, sosfiltfilt
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm


In [ ]:
# ── constants ────────────────────────────────────────────────────────────
FS          = 200
WIN_SAMPLES = 10_000
TARGET_SIZE = 512
CROP_LENGTHS = [2000, 5000, 10_000]
BANDPASS_LO  = 0.5
BANDPASS_HI  = 40.0
VOTE_COLS   = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
CLASS_NAMES = ['seizure', 'lpd', 'gpd', 'lrda', 'grda', 'other']

BIPOLAR_PAIRS = [
    ('Fp1', 'F7'), ('F7', 'T3'), ('T3', 'T5'), ('T5', 'O1'),
    ('Fp2', 'F8'), ('F8', 'T4'), ('T4', 'T6'), ('T6', 'O2'),
    ('Fp1', 'F3'), ('F3', 'C3'), ('C3', 'P3'), ('P3', 'O1'),
    ('Fp2', 'F4'), ('F4', 'C4'), ('C4', 'P4'), ('P4', 'O2'),
    ('Fz', 'Cz'), ('Cz', 'Pz'),
]
NEEDED_COLS = list(dict.fromkeys(c for pair in BIPOLAR_PAIRS for c in pair))

# ── signal processing ─────────────────────────────────────────────────────
def _load_eeg_window(eeg_id, offset_sec, eeg_dir):
    raw = pq.read_table(Path(eeg_dir) / f'{eeg_id}.parquet', columns=NEEDED_COLS).to_pandas()
    start = int(offset_sec * FS)
    window = raw.iloc[start: start + WIN_SAMPLES]
    window = window.interpolate(axis=0, limit_direction='both').fillna(0)
    return window.values.T.astype(np.float32)

def _bipolar_montage(eeg):
    ch_idx = {c: i for i, c in enumerate(NEEDED_COLS)}
    out = np.zeros((18, eeg.shape[1]), dtype=np.float32)
    for k, (a, b) in enumerate(BIPOLAR_PAIRS):
        out[k] = eeg[ch_idx[a]] - eeg[ch_idx[b]]
    return out

def _bandpass(signals):
    sos = butter(5, [BANDPASS_LO, BANDPASS_HI], btype='bandpass', fs=FS, output='sos')
    return sosfiltfilt(sos, signals, axis=-1).astype(np.float32)

def _signals_to_image(signals):
    channels = []
    for crop_len in CROP_LENGTHS:
        start = (signals.shape[1] - crop_len) // 2
        crop = signals[:, start: start + crop_len]
        ch = cv2.resize(crop.astype(np.float32), (TARGET_SIZE, TARGET_SIZE),
                        interpolation=cv2.INTER_LINEAR)
        channels.append(ch)
    img = np.stack(channels)
    return ((img - img.mean()) / (img.std() + 1e-6)).astype(np.float32)

# ── dataset ───────────────────────────────────────────────────────────────
class InferenceDataset(Dataset):
    def __init__(self, df, eeg_dir):
        self.df = df.reset_index(drop=True)
        self.eeg_dir = eeg_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        eeg = _load_eeg_window(int(row['eeg_id']), row['eeg_label_offset_seconds'], self.eeg_dir)
        bip = _bipolar_montage(eeg)
        bip = np.clip(bip, -1024.0, 1024.0) / 32.0
        filt = _bandpass(bip)
        img = _signals_to_image(filt)
        return torch.from_numpy(img)

# ── model ─────────────────────────────────────────────────────────────────
class EEGModel(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=False, num_classes=0)
        self.drop = nn.Dropout(0.0)
        self.fc = nn.Linear(self.backbone.num_features, 6)

    def forward(self, x):
        return F.log_softmax(self.fc(self.drop(self.backbone(x))), dim=1)

# ── device ────────────────────────────────────────────────────────────────
device = (
    torch.device('mps')  if torch.backends.mps.is_available() else
    torch.device('cuda') if torch.cuda.is_available() else
    torch.device('cpu')
)
print(f'device: {device}')


In [ ]:
df = pd.read_csv(CSV_PATH)
df_infer = df.groupby('eeg_id', as_index=False).first().reset_index(drop=True)

if 'eeg_label_offset_seconds' not in df_infer.columns:
    df_infer['eeg_label_offset_seconds'] = 0

dataset = InferenceDataset(df_infer, EEG_DIR)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=False)
print(f'{len(dataset)} EEG samples')
print(f'columns: {list(df_infer.columns)}')


In [ ]:
all_fold_preds = []

for ckpt_path in CKPT_PATHS:
    model = EEGModel(BACKBONE).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    model.eval()
    print(f'loaded  {Path(ckpt_path).name}')

    fold_preds = []
    with torch.no_grad():
        for imgs in tqdm(loader, desc=Path(ckpt_path).stem, leave=False):
            probs = model(imgs.to(device)).exp().cpu()
            fold_preds.append(probs)

    all_fold_preds.append(torch.cat(fold_preds))

preds = torch.stack(all_fold_preds).mean(dim=0).numpy()  # (N, 6)
print(f'preds: {preds.shape}')


In [ ]:
sub = pd.DataFrame(preds, columns=VOTE_COLS)
sub.insert(0, 'eeg_id', df_infer['eeg_id'].values)

sub.to_csv(OUT_CSV, index=False)
print(f'saved → {OUT_CSV}')
sub.head(10)
